# 04 · Phase 3: train the audio branch
ResNet-18 (ImageNet-pretrained) on log-mel spectrograms, two steps:
1. **Pre-train on ASVspoof 2019 LA**: lots of bona fide vs. synthetic/converted speech.
2. **Fine-tune on FakeAVCeleb audio**: adapts to the cloned voices in our main dataset.

- **Runtime:** T4 GPU. **Needs:** `processed/asvspoof/index.csv` (and `processed/fakeavceleb/`) from notebook 02.
- DFDC has no separate audio labels, so it is not used for this branch.
- **Disconnected?** Re-run setup + the same cell: training resumes from `last.pt`.

In [ ]:
# --- Session setup: run this first, every time you open the notebook ---
REPO_URL = "https://github.com/tanishq1101/robust-multimodal-deepfake.git"  # "" = use code copied to Drive

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    CODE = "/content/DeepFake"
    if os.path.exists(CODE):
        subprocess.run(["git", "-C", CODE, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE], check=True)
else:
    # No GitHub repo yet: upload the project folder to Google Drive at MyDrive/deepfake_project/code
    CODE = "/content/drive/MyDrive/deepfake_project/code"
    assert os.path.exists(CODE), f"Upload the project folder to {CODE} or set REPO_URL"

os.chdir(CODE)
!pip install -q -r requirements-colab.txt && pip install -q -e .
sys.path.insert(0, f"{CODE}/src")

from deepfake.config import load_config
cfg = load_config()
print("data_root (Drive):", cfg["data_root"])
print("raw data (VM disk):", cfg["paths"]["raw"])

In [ ]:
import json, os, pandas as pd
from pathlib import Path
from IPython.display import Image, display
CKPT = Path(cfg["paths"]["checkpoints"])

def show_run(run):
    run_dir = CKPT / run
    display(Image(str(run_dir / "curves.png")))
    m = json.loads((run_dir / "test" / "metrics.json").read_text())
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in m.items() if k not in ("index", "checkpoint")})
    display(Image(str(run_dir / "test" / "roc.png")))
    if (run_dir / "test" / "per_category.csv").exists():
        display(pd.read_csv(run_dir / "test" / "per_category.csv").round(4))

## Step 1: pre-train on ASVspoof 2019 LA

In [ ]:
!python -m deepfake.training.train_audio --dataset asvspoof --run-name audio_asvspoof --epochs 15 --batch-size 32 --workers 2

Results. `per_category` lists the **EER for each attack system**. The ASVspoof eval set contains
attacks (A07–A19) never seen in training, so this measures generalisation to new voice-cloning methods.

In [ ]:
show_run("audio_asvspoof")

## Step 2: fine-tune on FakeAVCeleb audio

In [ ]:
if os.path.exists(f"{cfg['paths']['processed']}/fakeavceleb/index.csv"):
    !python -m deepfake.training.train_audio --dataset fakeavceleb --run-name audio_fakeavceleb --init-from {CKPT}/audio_asvspoof/best.pt --lr 3e-5 --freeze-epochs 0 --epochs 10 --batch-size 32 --workers 2
else:
    print("FakeAVCeleb not preprocessed yet: run notebooks 01 + 02 once access is approved.")

In [ ]:
if (CKPT / "audio_fakeavceleb" / "test" / "metrics.json").exists():
    show_run("audio_fakeavceleb")

## Ablation: does ASVspoof pre-training help?
Train on FakeAVCeleb from ImageNet weights only and compare test AUC/EER with step 2.

In [ ]:
if os.path.exists(f"{cfg['paths']['processed']}/fakeavceleb/index.csv"):
    !python -m deepfake.training.train_audio --dataset fakeavceleb --run-name audio_fakeavceleb_noprt --epochs 10 --batch-size 32 --workers 2
    rows = []
    for run in ["audio_fakeavceleb_noprt", "audio_fakeavceleb"]:
        m = json.loads((CKPT / run / "test" / "metrics.json").read_text())
        rows.append({"run": run, "auc": m["auc"], "eer": m["eer"], "accuracy": m["accuracy"]})
    display(pd.DataFrame(rows).round(4))